In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Normalize
import timm
from torchmetrics.segmentation import DiceScore
from torchmetrics import JaccardIndex

import os
import glob
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from matplotlib.colors import ListedColormap
import pandas as pd
from collections import OrderedDict

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

d:\Dev\devtools\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.7.1+cu118
Using device: cuda


# Helper Function

In [3]:
def set_bn_eval(module):
    if isinstance(module, nn.BatchNorm2d):
        module.eval()

# LLEM and AFM

In [4]:
class CSDN_Tem(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(CSDN_Tem, self).__init__()
        self.depth_conv = nn.Conv2d(in_channels=in_ch, out_channels=in_ch, kernel_size=3, stride=1, padding=1, groups=in_ch)
        self.point_conv = nn.Conv2d(in_channels=in_ch, out_channels=out_ch, kernel_size=1, stride=1, padding=0, groups=1)
    def forward(self, input):
        return self.point_conv(self.depth_conv(input))

class EnhanceNet(nn.Module):
    def __init__(self):
        super(EnhanceNet, self).__init__()
        self.relu = nn.ReLU(inplace=True)
        number_f = 32
        self.e_conv1 = CSDN_Tem(3, number_f)
        self.e_conv2 = CSDN_Tem(number_f, number_f)
        self.e_conv3 = CSDN_Tem(number_f, number_f)
        self.e_conv4 = CSDN_Tem(number_f, number_f)
        self.e_conv5 = CSDN_Tem(number_f * 2, number_f)
        self.e_conv6 = CSDN_Tem(number_f * 2, number_f)
        self.e_conv7 = CSDN_Tem(number_f * 2, 3)
    def forward(self, x):
        x1 = self.relu(self.e_conv1(x))
        x2 = self.relu(self.e_conv2(x1))
        x3 = self.relu(self.e_conv3(x2))
        x4 = self.relu(self.e_conv4(x3))
        x5 = self.relu(self.e_conv5(torch.cat([x3, x4], 1)))
        x6 = self.relu(self.e_conv6(torch.cat([x2, x5], 1)))
        x_r = torch.tanh(self.e_conv7(torch.cat([x1, x6], 1)))
        return x1, x_r

In [5]:
# --- Attention Fusion Module ---
class AFM(nn.Module):
    def __init__(self, in_channels_llem, in_channels_seg):
        super(AFM, self).__init__()
        self.conv_llem = nn.Conv2d(in_channels_llem, in_channels_seg, kernel_size=1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.channel_attention = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_channels_seg, in_channels_seg // 16, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(in_channels_seg // 16, in_channels_seg, 1, bias=False), nn.Sigmoid())
    def forward(self, llem_features, seg_features):
        llem_features = self.conv_llem(llem_features)
        fused_features = self.relu(llem_features + seg_features)
        attention = self.channel_attention(fused_features)
        return fused_features * attention

# DeepLabv3+

In [6]:
class ECALayer(nn.Module):
    def __init__(self, channel, k_size=3):
        super(ECALayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)

class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(SEBlock, self).__init__()
        self.global_avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        y = self.global_avgpool(x)
        y = self.fc1(y)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y)
        return x * y

class CBAMLayer(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(CBAMLayer, self).__init__()
        self.channel_attention = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False), nn.Sigmoid())
        self.spatial_attention = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())
    def forward(self, x):
        ca = self.channel_attention(x); x = x * ca
        sa_input = torch.cat([torch.mean(x, dim=1, keepdim=True), torch.max(x, dim=1, keepdim=True)[0]], dim=1); sa = self.spatial_attention(sa_input); x = x * sa
        return x

class FCALayer(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(FCALayer, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        y = self.global_avg_pool(x); y = self.fc1(y); y = self.relu(y); y = self.fc2(y); y = self.sigmoid(y)
        return x * y

# --- Backbone dan Modul DeepLabV3+ (Sesuai dengan Notebook Training Anda) ---
class ResNet18Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super(ResNet18Backbone, self).__init__()
        self.resnet18 = timm.create_model('resnet18', pretrained=pretrained, features_only=True, out_indices=(0, 1, 2, 3))
        self.resnet18.apply(set_bn_eval)
        self.eca = ECALayer(channel=256, k_size=3)
        self.se = SEBlock(in_channels=256)
        self.cbam = CBAMLayer(in_channels=256)
    def forward(self, x):
        features = self.resnet18(x)
        low_level_feature = features[1]
        high_level_feature = self.eca(features[-1])
        high_level_feature = self.se(high_level_feature)
        high_level_feature = self.cbam(high_level_feature)
        return {'low_level': low_level_feature, 'out': high_level_feature, 'features': features}

class AtrousSeparableConvolution(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True):
        super(AtrousSeparableConvolution, self).__init__()
        self.body = nn.Sequential(nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=bias, groups=in_channels), nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=bias))
    def forward(self, x):
        return self.body(x)

class ASPPPooling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ASPPPooling, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        size = x.shape[-2:]; x = self.avg_pool(x); x = self.conv(x)
        if x.shape[-1] > 1 and x.shape[-2] > 1: x = self.bn(x)
        x = self.relu(x); return F.interpolate(x, size=size, mode='bilinear', align_corners=False)

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, atrous_rates):
        super(ASPP, self).__init__()
        modules = [nn.Sequential(nn.Conv2d(in_channels, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))]
        for rate in atrous_rates: modules.append(AtrousSeparableConvolution(in_channels, out_channels, kernel_size=3, padding=rate, dilation=rate, bias=False))
        modules.append(ASPPPooling(in_channels, out_channels)); self.convs = nn.ModuleList(modules)
        self.project = nn.Sequential(nn.Conv2d(len(modules) * out_channels, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.Dropout(0.1))
    def forward(self, x):
        res = [conv(x) for conv in self.convs]; res = torch.cat(res, dim=1); return self.project(res)

class FEM(nn.Module):
    def __init__(self, in_channels_list, out_channels):
        super(FEM, self).__init__()
        self.convs = nn.ModuleList([nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)) for in_channels in in_channels_list])
        self.output_conv = nn.Sequential(nn.Conv2d(len(in_channels_list) * out_channels, out_channels, kernel_size=1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
    def forward(self, features):
        res = []; target_size = features[-1].shape[2:]
        for conv, feature in zip(self.convs, features): feature = F.interpolate(feature, size=target_size, mode='bilinear', align_corners=False); res.append(conv(feature))
        res = torch.cat(res, dim=1); return self.output_conv(res)

class DeepLabHeadV3Plus(nn.Module):
    def __init__(self, in_channels, low_level_channels, num_classes, aspp_dilate):
        super(DeepLabHeadV3Plus, self).__init__()
        self.project = nn.Sequential(nn.Conv2d(low_level_channels, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.cbam = CBAMLayer(in_channels=256)
        self.aspp = ASPP(in_channels, 256, aspp_dilate)
        self.cbam_decoder = CBAMLayer(in_channels=304)
        self.fca = FCALayer(in_channels=304)
        self.classifier = nn.Sequential(AtrousSeparableConvolution(304, 256, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Conv2d(256, num_classes, 1))
    def forward(self, feature, afm_features=None):
        low_level_feature = self.project(feature['low_level'])
        output_feature = self.cbam(feature['out'])
        output_feature = self.aspp(output_feature)
        if afm_features is not None:
            output_feature += afm_features
        output_feature = F.interpolate(output_feature, size=low_level_feature.shape[2:], mode='bilinear', align_corners=False)
        concatenated_features = torch.cat([low_level_feature, output_feature], dim=1)
        concatenated_features = self.cbam_decoder(concatenated_features)
        concatenated_features = F.interpolate(concatenated_features, scale_factor=4, mode='bilinear', align_corners=False)
        concatenated_features = self.fca(concatenated_features)
        return self.classifier(concatenated_features)
        
class DeepLabV3Plus(nn.Module):
    def __init__(self, backbone, num_classes, output_stride=8):
        super(DeepLabV3Plus, self).__init__()
        atrous_rates = [12, 24, 36] if output_stride == 8 else [6, 12, 18]
        self.backbone = backbone
        self.head = DeepLabHeadV3Plus(in_channels=256, low_level_channels=64, num_classes=num_classes, aspp_dilate=atrous_rates)
        self.fem = FEM(in_channels_list=[64, 64, 128, 256], out_channels=256)
    def forward(self, x):
        backbone_features = self.backbone(x)
        fem_features = self.fem(backbone_features['features'])
        output = self.head({'low_level': backbone_features['low_level'], 'out': backbone_features['out']}, afm_features=fem_features)
        output = F.interpolate(output, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return output, fem_features

# End-to-End Architecture

In [7]:
class FLLENet_EndToEnd(nn.Module):
    def __init__(self, num_classes, output_stride=8):
        super(FLLENet_EndToEnd, self).__init__()
        self.llem = EnhanceNet()
        backbone = ResNet18Backbone(pretrained=True)
        self.segmentation_network = DeepLabV3Plus(backbone, num_classes=num_classes, output_stride=output_stride)
        self.afm = AFM(in_channels_llem=32, in_channels_seg=256)
        self.normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def enhance_image(self, x, x_r, n_iterations):
        enhanced_image = x
        for _ in range(n_iterations):
            enhanced_image = enhanced_image + x_r * (torch.pow(enhanced_image, 2) - enhanced_image)
        return torch.clamp(enhanced_image, 0, 1)

    def forward(self, x, n_iterations=4):
        llem_features, x_r = self.llem(x)
        enhanced_image = self.enhance_image(x, x_r, n_iterations=n_iterations)
        normalized_enhanced_image = self.normalize(enhanced_image)
        backbone_output = self.segmentation_network.backbone(normalized_enhanced_image)
        fem_features = self.segmentation_network.fem(backbone_output['features'])

        llem_features_resized = F.interpolate(llem_features, size=fem_features.shape[2:], mode='bilinear', align_corners=False)
        
        afm_output = self.afm(llem_features_resized, fem_features)
        
        head_input_features = {'low_level': backbone_output['low_level'], 'out': backbone_output['out']}
        output_mask = self.segmentation_network.head(head_input_features, afm_features=afm_output)
        output_mask = F.interpolate(output_mask, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return output_mask, enhanced_image

# Utils Function

In [8]:
class CustomDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths; self.mask_paths = mask_paths; self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx]); image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        if self.transform: augmented = self.transform(image=image, mask=mask); image = augmented['image']; mask = augmented['mask']
        return image.float() / 255.0, mask.long()

In [9]:
import time

IMAGE_HEIGHT = 256
IMAGE_WIDTH = 448

baseline_inference_alb_tf = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])
endtoend_inference_alb_tf = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    # A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

def baseline_preprocess(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    out = baseline_inference_alb_tf(image=frame)
    tensor = out["image"]
    return tensor.unsqueeze(0) 

def endtoend_preprocess(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    out = endtoend_inference_alb_tf(image=frame)
    tensor = out["image"].float() / 255.0
    return tensor.unsqueeze(0) 

In [10]:
NUM_CLASSES = 4
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 448
CLASS_NAMES = ['background', 'road', 'lm_solid', 'lm_dashed']

TEST_IMG_PATH = 'dataset/v6-300-tvt/v6-300-tvt.voc/test/images'
TEST_MASK_PATH = 'dataset/v6-300-tvt/v6-300-tvt.voc/test/masks'
MODEL_WEIGHTS_DIR='models'

test_transform = A.Compose([A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH), ToTensorV2()])
# normalize_transform = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

test_img_paths = sorted(glob.glob(TEST_IMG_PATH))
test_mask_paths = sorted(glob.glob(TEST_MASK_PATH))

In [12]:
end_to_end_model = FLLENet_EndToEnd(num_classes=NUM_CLASSES).to(device)
END_TO_END_MODEL_PATH = 'models/end2end.pth'
BASELINE_MODEL_PATH = 'models/baseline.pth'

if not test_img_paths:
    print("!!! WARNING: No testing images found. Configure the path name correctly.")
else:
    test_dataset = CustomDataset(test_img_paths, test_mask_paths, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

    # --- Memuat Model End-to-End ---
    print("Loading End-to-End Model...")
    end_to_end_model = FLLENet_EndToEnd(num_classes=NUM_CLASSES).to(device)
    end_to_end_model.load_state_dict(torch.load(END_TO_END_MODEL_PATH, map_location=device))
    print("End-to-End Model loaded successfully.")

    # --- Memuat Model Baseline (Hanya Segmentasi) ---
    print("\nLoading Baseline Model (segmentation only)...")
    baseline_backbone = ResNet18Backbone(pretrained=False) # Tidak perlu bobot pretrained ImageNet
    baseline_model = DeepLabV3Plus(baseline_backbone, num_classes=NUM_CLASSES).to(device)
    
    # Ekstrak bobot bagian segmentasi dari model end-to-end
    full_state_dict = torch.load(END_TO_END_MODEL_PATH, map_location=device)
    baseline_state_dict = torch.load(BASELINE_MODEL_PATH, map_location=device)
    
    baseline_model.load_state_dict(baseline_state_dict)
    
    print("Baseline Model weight successfully loaded with the weight from end-to-end model.")


Loading End-to-End Model...
End-to-End Model loaded successfully.

Loading Baseline Model (segmentation only)...
Baseline Model weight successfully loaded with the weight from end-to-end model.


In [12]:
video_path = "dataset/Trim ahmad yani.mp4" 

def evaluate_fps(model, name):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    total_time = 0.0

    model.eval()
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # ---- select preprocess function ----
            if "baseline" in name.lower():
                inp = baseline_preprocess(frame)
            else:
                inp = endtoend_preprocess(frame)
            # ------------------------------------------------

            inp = inp.to(device)

            # safety check
            if inp.dim() != 4:
                inp = inp.unsqueeze(0)

            start = time.time()
            _ = model(inp)
            end = time.time()

            total_time += (end - start)
            frame_count += 1

    cap.release()
    mean_fps = frame_count / total_time if total_time > 0 else 0
    print(f"[{name}] Frames: {frame_count}, Mean FPS: {mean_fps:.2f}")
    return mean_fps

# ---- Run tests ----
baseline_fps = evaluate_fps(baseline_model, "Baseline Model")
end2end_fps = evaluate_fps(end_to_end_model, "End-to-End Model")

print("\n===== RESULTS =====")
print(f"Baseline Mean FPS   : {baseline_fps:.2f}")
print(f"End-to-End Mean FPS : {end2end_fps:.2f}")

[Baseline Model] Frames: 598, Mean FPS: 57.79
[End-to-End Model] Frames: 598, Mean FPS: 31.15

===== RESULTS =====
Baseline Mean FPS   : 57.79
End-to-End Mean FPS : 31.15


In [13]:
video_path = "dataset/Trim jalan CPI gelap Fix.mp4"  # change this to your source video

# ---- Run tests ----
baseline_fps = evaluate_fps(baseline_model, "Baseline Model")
end2end_fps = evaluate_fps(end_to_end_model, "End-to-End Model")

print("\n===== RESULTS =====")
print(f"Baseline Mean FPS   : {baseline_fps:.2f}")
print(f"End-to-End Mean FPS : {end2end_fps:.2f}")

[Baseline Model] Frames: 182, Mean FPS: 63.28
[End-to-End Model] Frames: 182, Mean FPS: 31.17

===== RESULTS =====
Baseline Mean FPS   : 63.28
End-to-End Mean FPS : 31.17


In [17]:
def mask_to_overlay(mask, frame):
    """
    mask: [H,W] int64 — class index
    frame: original BGR frame
    """

    mask = cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]))

    # Color palette (change if needed)
    colors = {
        0: (0, 0, 0),  # background
        1: (255, 0, 0),  # road
        2: (0, 255, 0),  # lane solid
        3: (0, 255, 255),  # lane dashed
    }

    overlay = np.zeros_like(frame)
    for cls, color in colors.items():
        overlay[mask == cls] = color

    blended = cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)
    return blended


# def run_and_save_video(model, output_path, show_fps=True):
#     cap = cv2.VideoCapture(video_path)

#     if not cap.isOpened():
#         print("ERROR: Cannot open video:", video_path)
#         return 0

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

#     writer = cv2.VideoWriter(
#         output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
#     )

#     frame_count = 0
#     total_time = 0.0

#     model.eval()
#     with torch.no_grad():
#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 print("Finished or failed to read frames.")
#                 break

#             # ---- select preprocess function ----
#             if "baseline" in output_path.lower():
#                 inp = baseline_preprocess(frame)
#             else:
#                 inp = endtoend_preprocess(frame)
#             # ------------------------------------------------
#             inp = inp.to(device)

#             start = time.time()
#             out = model(inp)
#             end = time.time()

#             # Handle tuple outputs (DeepLab-like)
#             logits = out[0] if isinstance(out, tuple) else out

#             # Argmax prediction
#             pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

#             # Visualization
#             vis = mask_to_overlay(pred, frame)
#             writer.write(vis)

#             total_time += end - start
#             frame_count += 1

#     cap.release()
#     writer.release()

#     mean_fps = frame_count / total_time if total_time > 0 else 0

#     if show_fps:
#         print(f"Saved: {output_path}")
#         print(f"Frames processed: {frame_count}")
#         print(f"Mean FPS: {mean_fps:.2f}")

#     return mean_fps

In [13]:
def run_and_save_video(model, output_path, video_path, max_frames=None, show_fps=True):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("ERROR: Cannot open video:", video_path)
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # --- Auto-detect device from the model ---
    device = next(model.parameters()).device 
    # ----------------------------------------
    
    if show_fps and max_frames:
        print(f"Processing capped at {max_frames} frames.")

    writer = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )

    frame_count = 0
    total_time = 0.0

    model.eval()
    with torch.no_grad():
        while True:
            # --- Check limit before processing ---
            if max_frames is not None and frame_count >= max_frames:
                print(f"Reached limit of {max_frames} frames.")
                break
            # -------------------------------------

            ret, frame = cap.read()
            if not ret:
                print("Finished or failed to read frames.")
                break

            if "baseline" in output_path.lower():
                inp = baseline_preprocess(frame)
            else:
                inp = endtoend_preprocess(frame)
            
            # Move input to the same device as the model
            inp = inp.to(device)

            start = time.time()
            out = model(inp)
            end = time.time()

            logits = out[0] if isinstance(out, tuple) else out
            pred = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

            vis = mask_to_overlay(pred, frame)
            writer.write(vis)

            total_time += end - start
            frame_count += 1

    cap.release()
    writer.release()

    mean_fps = frame_count / total_time if total_time > 0 else 0

    if show_fps:
        print(f"Saved: {output_path}")
        print(f"Frames processed: {frame_count}")
        print(f"Mean FPS: {mean_fps:.2f}")

    return mean_fps

# Video Inference 1

In [28]:
# -------------------------------
# Choose which model to run
# -------------------------------
baseline_model = baseline_model  
model = end_to_end_model  
model_name = "End-to-End"
baseline_model_name = "Baseline"

# -------------------------------
# Input & Output paths
# -------------------------------
video_path = "dataset/Trim ahmad yani.mp4"
output_path = f"inference/{model_name.replace('-', '').lower()}_inference.mp4"
baseline_output_path = f"inference/{baseline_model_name.replace('-', '').lower()}_inference.mp4"

In [29]:
# run_and_save_video(baseline_model, baseline_output_path)
run_and_save_video(
    model=baseline_model, 
    output_path=baseline_output_path,
    video_path=video_path,
    max_frames=182
)

# run_and_save_video(model, output_path)
run_and_save_video(
    model=model, 
    output_path=output_path, 
    video_path=video_path, 
    max_frames=182
)

Processing capped at 182 frames.
Reached limit of 182 frames.
Saved: inference/baseline_inference.mp4
Frames processed: 182
Mean FPS: 69.46
Processing capped at 182 frames.
Reached limit of 182 frames.
Saved: inference/endtoend_inference.mp4
Frames processed: 182
Mean FPS: 26.03


26.03319306313373

# Video Inference 2

In [ ]:
# -------------------------------
# Choose which model to run
# -------------------------------
baseline_model = baseline_model  
model = end_to_end_model  
model_name = "End-to-End"
baseline_model_name = "Baseline"

# -------------------------------
# Input & Output paths
# -------------------------------
video_path = "dataset/Trim jalan CPI gelap Fix.mp4"
output_path = f"inference/{model_name.replace('-', '').lower()}_CPI_inference.mp4"
baseline_output_path = f"inference/{baseline_model_name.replace('-', '').lower()}_CPI_inference.mp4"

In [23]:
# run_and_save_video(baseline_model, baseline_output_path)
run_and_save_video(
    model=baseline_model, 
    output_path=baseline_output_path,
    video_path=video_path,
    max_frames=182
)

# run_and_save_video(model, output_path)
run_and_save_video(
    model=model, 
    output_path=output_path,
    video_path=video_path,
    max_frames=182
)

Processing capped at 182 frames.
Reached limit of 182 frames.
Saved: inference/baseline_CPI_inference.mp4
Frames processed: 182
Mean FPS: 57.37
Processing capped at 182 frames.
Reached limit of 182 frames.
Saved: inference/endtoend_CPI_inference.mp4
Frames processed: 182
Mean FPS: 23.03


23.025234263001465

# Video Inference 3

In [26]:
# -------------------------------
# Choose which model to run
# -------------------------------
baseline_model = baseline_model  
model = end_to_end_model  
model_name = "End-to-End"
baseline_model_name = "Baseline"

# -------------------------------
# Input & Output paths
# -------------------------------
video_path = "dataset/Fix Nusantara.mp4"
output_path = f"inference/{model_name.replace('-', '').lower()}_Nusantara_inference.mp4"
baseline_output_path = f"inference/{baseline_model_name.replace('-', '').lower()}_Nusantara_inference.mp4"

In [27]:
# run_and_save_video(baseline_model, baseline_output_path)
run_and_save_video(
    model=baseline_model, 
    output_path=baseline_output_path,
    video_path=video_path,
    max_frames=182
)

# run_and_save_video(model, output_path)
run_and_save_video(
    model=model, 
    output_path=output_path,
    video_path=video_path,
    max_frames=182
)

Processing capped at 182 frames.
Reached limit of 182 frames.
Saved: inference/baseline_Nusantara_inference.mp4
Frames processed: 182
Mean FPS: 67.57
Processing capped at 182 frames.
Reached limit of 182 frames.
Saved: inference/endtoend_Nusantara_inference.mp4
Frames processed: 182
Mean FPS: 26.10


26.09737585356433